### 연습문제
- Doc2Vec를 이용하여 감성 분석
- 데이터는 ratings_train.txt 파일을 로드
    - 특수 문자, 2칸 이상의 공백의 문자를 제거하는 정규화
    - document 컬럼의 데이터에서 중복 데이터를 제거
    - 빈 텍스트가 존재한다면 해당 데이터 제거
    - 상위 5000개 데이터를 학습 데이터로 이용
- 토큰화 함수는 Komoran을 사용
    - 품사 필터 : NNP, NNG, VV, VA, MAG, XR 만을 사용
    - 불용어 단어 : 하다, 되다, 이다, 것, 수, 거 단어들을 제외
- 독립변수(document), 종속변수(label) 데이터를 나눠주고 train, test로 데이터를 분할(8:2)
- Doc2Vec 객체를 생성하여 벡터화
    - 매개변수
        - vetor_size = 200
        - window = 5
        - min_count = 2
        - dm = 1
        - negative = 5
        - seed = 42
        - epochs = 50
    - 학습 데이터는 train 데이터를 이용
- Doc2Vec 객체에서 train, test 데이터를 infer_vector() 함수를 이용하여 벡터 데이터를 생성
- ML 분류 모델을 이용하여 임베딩된 데이터를 독립변수로 사용하여 학습
    - 정확도를 확인
    - Logistic
        - max_iter = 2000
        - random_state = 42
    - LinearSVC
        - random_state = 42
- test데이터를 이용하여 2개의 모델 중 정확도 높은 모델을 검색

In [2]:
import pandas as pd
import re
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from konlpy.tag import Komoran

In [3]:
df = pd.read_csv('../data/ratings_train.txt', sep = '\t')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype
---  ------    --------------   -----
 0   id        150000 non-null  int64
 1   document  149995 non-null  str  
 2   label     150000 non-null  int64
dtypes: int64(2), str(1)
memory usage: 3.4 MB


In [4]:
df.dropna(inplace = True)

In [5]:
# 텍스트 정규화 함수
def nomalize(text):
    text = re.sub(r'[^가-힣0-9a-zA-z\s\.]', ' ', str(text))
    text = re.sub(r'\s+', ' ',text).strip()
    return text

In [6]:
df2 = df.copy()

In [7]:
# document 컬럼의 데이터를 정규화
df['document'] = df['document'].map(nomalize)

In [8]:
df2.apply(lambda x : print(x))

0          9976970
1          3819312
2         10265843
3          9045019
4          6483659
            ...   
149995     6222902
149996     8549745
149997     9311800
149998     2376369
149999     9619869
Name: id, Length: 149995, dtype: int64
0                                       아 더빙.. 진짜 짜증나네요 목소리
1                         흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나
2                                         너무재밓었다그래서보는것을추천한다
3                             교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정
4         사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...
                                ...                        
149995                                  인간이 문제지.. 소는 뭔죄인가..
149996                                        평점이 너무 낮아서...
149997                      이게 뭐요? 한국인은 거들먹거리고 필리핀 혼혈은 착하다?
149998                          청춘 영화의 최고봉.방황과 우울했던 날들의 자화상
149999                             한국 영화 최초로 수간하는 내용이 담긴 영화
Name: document, Length: 149995, dtype: str
0         0
1         1
2         0
3         0
4

id          None
document    None
label       None
dtype: object

In [9]:
# 공백 데이터가 존재하는가? -> 정규화를 통해서 '  ' -> ' '이 작업 후 strip()을 사용 -> ''
df = df.loc[
    ~(df['document'] == ''),

]

In [10]:
# 중복 데이터 제거
df.drop_duplicates('document',inplace = True)

In [12]:
allow_pos = ['NNP','NNG','VV','VA','MAG','XG']
stop_word = ['하다','되다','이다','것','수','거']

komoran = Komoran()

def tokenize(text):
    tokens = []
    for word, pos in komoran.pos(text):
        if pos in allow_pos and word not in stop_word:
            tokens.append(word)
    return tokens

In [13]:
df3 = df.head(5000)

In [15]:
tokenize_sentence = [
    tokenize(text) for text in df3['document'].values
]
tokenize_sentence

[['더빙', '진짜', '짜증', '나', '목소리'],
 ['포스터', '초딩', '영화', '오버', '연기', '가볍'],
 [],
 ['교도소', '이야기', '솔직히', '재미', '없', '평점', '조정'],
 ['익살', '연기', '돋보이', '영화', '스파이더맨', '늙', '보이', '하', '커스틴 던스트', '너무나'],
 ['막', '걸음마', '떼', '초등학교', '학년', '용', '영화', '별', '반개', '아깝'],
 ['원작', '긴장감', '제대로', '살리'],
 ['반개',
  '아깝',
  '욕',
  '나오',
  '이응경',
  '길용우',
  '연기',
  '생활',
  '이',
  '정말',
  '발로',
  '납치',
  '감금',
  '반복',
  '반복',
  '이',
  '드라마',
  '가족',
  '없',
  '연기',
  '못하',
  '사람',
  '모이'],
 ['액션', '없', '재미', '있', '안', '영화'],
 ['왜', '평점', '낮', '꽤', '보', '헐리우드', '너무', '길들이', '있'],
 [],
 ['볼', '때', '눈물', '나서', '죽', '향수', '자극', '허진호', '감성', '절제', '멜로', '달인'],
 ['울', '손들', '횡단보도', '건너', '때', '뛰쳐나오', '이범수', '연기', '드럽'],
 ['좋', '신문', '기사', '로만', '보다', '보', '자꾸', '잊어버리', '사람'],
 ['취향',
  '존중',
  '진짜',
  '극장',
  '보',
  '영화',
  '가장',
  '노',
  '재',
  '노',
  '감동',
  '스토리',
  '어거지',
  '감동',
  '어거지'],
 ['매번', '긴장'],
 ['참',
  '사람',
  '웃기',
  '바스코',
  '이기',
  '락스',
  '코',
  '까',
  '고',
  '바비',
  '이기',
  '아이돌',
  '깔',
  '그냥',

In [16]:
X = tokenize_sentence
y = df3['label'].values

In [17]:
# train, test 데이터셋을 분할
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [18]:
df3['label'].value_counts()

label
0    2504
1    2496
Name: count, dtype: int64

In [ ]:
# 문서에 Tag 부착
# 빈 토큰 리스트를 제외하고 태그를 부착
def tagged_docs(token_data):
    tagged = []
    for idx, toks in enumerate(token_data):
        # toks의 길이가 0이라면 -> 학습에서 큰 의미가 없음 제외
        if len(toks) == 0:
            continue
        tagged.append(
            TaggedDocument(
                words = toks,
                tags = [f'DOC_{idx}']
            )
        )
    return tagged

In [21]:
X_train_tag = tagged_docs(X_train)
print(len(X_train_tag), len(X_train))

3922 4000


In [22]:
# Doc2Vec 객체를 생성
model = Doc2Vec(
    documents = X_train_tag,
    vector_size=200,
    window=2,
    min_count=5,
    negative = 5,
    seed = 42,
    epochs = 59
)

In [23]:
# Doc2Vec 객체를 먼저 생성하고 추후에 학습
model2 = Doc2Vec(
    vector_size=200,
    window = 5,
    min_count = 2, 
    negative = 5,
    seed = 42,
    epochs = 50
)
# 단어 사전 생성
model2.build_vocab(X_train_tag)
# 학습
model2.train(
    X_train_tag,total_examples=len(X_train_tag), epochs=50
)

In [25]:
# 2개의 모델에서 단어 사전의 개수를 확인
print(len(model.wv))
print(len(model2.wv))

1042
2722


In [26]:
print(len(model.dv))

3922


In [28]:
def infer_vector(model, norm_tokens, epochs = 50):
    # norm_texts : 텍스트 정규화가 끝나고 토큰화가 완료된 데이터
    # model : 임베딩 모델

    result = []

    for tokens in norm_tokens:
        # tokens의 길이가 0이라면 0 행렬로 되돌려둔다
        if len(tokens) == 0:
            result.append(
                np.zeros(model.vector_size, dtype = np.float32)
            )
        else:
            vec = model.infer_vector(tokens, epochs = epochs)
            result.append(vec)
    return np.array(result)

In [29]:
X_train_vec = infer_vector(model, X_train)
X_test_vec = infer_vector(model, X_test)

In [30]:
X_train_vec.shape

(4000, 200)

In [31]:
def eval_clf(model, X_train, X_test, y_train, y_test):
    # model : 분류 모델 입력
    model.fit(X_train, y_train)

    pred = model.predict(X_test)

    print(classification_report(pred, y_test))

In [32]:
logi = LogisticRegression(max_iter=2000, random_state=42)
svc = LinearSVC(random_state=42)

In [33]:
eval_clf(logi, X_train_vec, X_test_vec, y_train, y_test)
eval_clf(svc, X_train_vec, X_test_vec, y_train, y_test)

              precision    recall  f1-score   support

           0       0.77      0.73      0.75       528
           1       0.72      0.76      0.74       472

    accuracy                           0.75      1000
   macro avg       0.75      0.75      0.75      1000
weighted avg       0.75      0.75      0.75      1000

              precision    recall  f1-score   support

           0       0.77      0.73      0.75       531
           1       0.71      0.76      0.73       469

    accuracy                           0.74      1000
   macro avg       0.74      0.74      0.74      1000
weighted avg       0.74      0.74      0.74      1000



In [34]:
# DataFrame을 train, test로 분할
train_df, test_df = train_test_split(
    df, test_size=0.2,random_state=42, stratify=df['label']
)

In [35]:
train_df['label'].value_counts()

label
0    58300
1    57598
Name: count, dtype: int64

In [36]:
test_df['label'].value_counts()

label
0    14575
1    14400
Name: count, dtype: int64